# Main 04 — longitudinal paired and clinical analyses

Run iTBS, rTMS, and sham paired analyses plus prespecified clinical correlations.

- Formal analysis universe: 12 cortical + 14 limbic ROIs (26 nodes)
- Cross-network connectivity: 12 × 14 = 168 edges
- Pipeline: `main`; step 4 of 10
- Run mode: standalone; all prerequisite steps load automatically in the same kernel.


In [1]:
from pathlib import Path
import os

_repo = Path(os.environ.get("METAFL_ROOT", Path.cwd())).resolve()
if not (_repo / "pipeline_steps").is_dir() and (_repo.parent / "pipeline_steps").is_dir():
    _repo = _repo.parent
if not (_repo / "pipeline_steps").is_dir():
    raise FileNotFoundError("Set METAFL_ROOT to the repository root")
os.environ["METAFL_ROOT"] = str(_repo)
os.chdir(_repo)

def _run_step(relative_path):
    _path = _repo / "pipeline_steps" / relative_path
    _code = compile(_path.read_text(encoding="utf-8"), str(_path), "exec")
    exec(_code, get_ipython().user_ns)
_run_step("main_01_setup_load.py")
_run_step("main_02_demographics_hc_reference.py")
_run_step("main_03_baseline_discovery_sensitivity.py")


MetaFL package root: .
Main-analysis output root: ./outputs/main_analyses
Cortical IDs: [3, 4, 5, 6, 19, 20, 21, 22, 25, 26, 31, 32]
Limbic IDs: [41, 42, 43, 44, 45, 46, 75, 76, 77, 78, 79, 80, 151, 152]
Cross-edge family: 168


,cohort,clinical_n,ses01_complete_n
0,HC,23,23
1,nTRD,25,25
2,TRD1,35,35
3,TRD2,104,104


,cohort,baseline_n,metric
0,nTRD,25,age/sex-adjusted normative-SC MwC deviation z
1,TRD1,35,age/sex-adjusted normative-SC MwC deviation z
2,TRD2,104,age/sex-adjusted normative-SC MwC deviation z


Written: ./outputs/main_analyses/demographic_and_clinical_summary.xlsx


,variable,HC,nTRD,TRD1,TRD2,primary_test,primary_p_value,TRD1_vs_TRD2_test,TRD1_vs_TRD2_p_value
0,Age,50.91 (17.04); n=23,52.32 (12.22); n=25,48.20 (15.52); n=35,39.09 (13.96); n=104,"one-way ANOVA: HC, nTRD, TRD1",0.561396,Welch t-test,0.003261
1,"Female, n (%)",14 (60.87%); n=23,14 (56.00%); n=25,25 (71.43%); n=35,73 (70.19%); n=104,"Pearson chi-square: HC, nTRD, TRD1",0.443311,Fisher exact test,1.000000
2,Education,14.17 (3.46); n=23,13.44 (3.86); n=25,13.14 (3.83); n=35,14.57 (2.81); n=104,"one-way ANOVA: HC, nTRD, TRD1",0.587829,Welch t-test,0.048544


...

,variable,note
0,MSM scores,Not required for the released analyses


,variable,iTBS,rTMS,sham,test,statistic,df_1,df_2,p_value
0,HDRS reduction (baseline - post),6.73 (8.20); n=37,7.65 (6.95); n=34,6.06 (4.94); n=33,ordinary one-way ANOVA,0.448348,2,101,0.639948


,variable,rTMS,iTBS,sham,test,p_value
0,"Age, years",39.35 (11.79); n=34,37.95 (14.52); n=37,40.09 (15.62); n=33,one-way ANOVA,0.809492
1,"Female, n (%)",23 (67.65%); n=34,27 (72.97%); n=37,23 (69.70%); n=33,Pearson chi-square,0.884317
2,"Education, years",14.53 (2.60); n=34,14.27 (2.59); n=37,14.94 (3.27); n=33,one-way ANOVA,0.611602
3,Baseline HDRS-17,22.74 (4.69); n=34,23.41 (5.53); n=37,22.30 (5.12); n=33,one-way ANOVA,0.663926
4,HDRS-17 reduction,7.65 (6.95); n=34,6.73 (8.20); n=37,6.06 (4.94); n=33,one-way ANOVA,0.639948


,modality,tested_n,primary_p_lt_0p01,HDRS_adjusted_p_lt_0p01,retained,lost,new
0,edge,168,22,2,2,20,0
1,regional,26,7,2,2,5,0
2,MwC,26,2,1,0,2,1


TRD2 subject-level deviations: (104, 168) (104, 26)


## Preamble


In [2]:
"""Auto-extracted pipeline step: Main 04 — longitudinal paired tests and clinical associations.

Do not edit generated copies independently of the source notebooks.
"""


'Auto-extracted pipeline step: Main 04 — longitudinal paired tests and clinical associations.\n\nDo not edit generated copies independently of the source notebooks.\n'

## Source 21


In [3]:
def add_hdrs_correlations(selected, value_matrix, feature_lookup, hdrs, age, sex, q_column):
    rows = []
    for _, selected_row in selected.iterrows():
        index = feature_lookup(selected_row)
        r_value, p_value, n_used = partial_spearman_age_sex(
            value_matrix[:, index], hdrs, age, sex
        )
        row = selected_row.to_dict()
        row.update({
            "HDRS_partial_Spearman_r": r_value,
            "HDRS_p_uncorrected": p_value,
            "n_used": n_used,
        })
        rows.append(row)

    columns = list(selected.columns) + [
        "HDRS_partial_Spearman_r", "HDRS_p_uncorrected", "n_used", q_column,
        "HDRS_primary_FDR_q_lt_0p05", "HDRS_exploratory_uncorrected_p_lt_0p05",
    ]
    if not rows:
        return pd.DataFrame(columns=columns)

    result = pd.DataFrame(rows)
    result[q_column] = bh_fdr(result["HDRS_p_uncorrected"])
    result["HDRS_primary_FDR_q_lt_0p05"] = result[q_column] < FDR_THRESHOLD
    result["HDRS_exploratory_uncorrected_p_lt_0p05"] = (
        result["HDRS_p_uncorrected"] < CLINICAL_P_THRESHOLD
    )
    return result.sort_values([q_column, "HDRS_p_uncorrected"]).reset_index(drop=True)


def save_prepost_and_hdrs(all_tests, selected, hdrs_results, output_dir, stem, q_column, include_hdrs=True):
    all_tests.to_excel(output_dir / f"{stem}_pairttest_all.xlsx", index=False)
    selected.to_excel(
        output_dir / f"{stem}_pairttest_uncorrected_p_lt_0p05.xlsx", index=False
    )
    hdrs_fdr = hdrs_results[hdrs_results["HDRS_primary_FDR_q_lt_0p05"]].copy()
    hdrs_exploratory = hdrs_results[
        hdrs_results["HDRS_exploratory_uncorrected_p_lt_0p05"]
    ].copy()
    if include_hdrs:
        hdrs_results.to_excel(
            output_dir / f"{stem}_HDRS_partial_spearman_all_prepost_p_lt_0p05.xlsx",
            index=False,
        )
        hdrs_fdr.to_excel(
            output_dir / f"{stem}_HDRS_partial_spearman_FDR_q_lt_0p05_primary.xlsx",
            index=False,
        )
        hdrs_exploratory.to_excel(
            output_dir / f"{stem}_HDRS_partial_spearman_uncorrected_p_lt_0p05_exploratory.xlsx",
            index=False,
        )
    return hdrs_fdr, hdrs_exploratory


def paired_mean_ci95(delta):
    delta = np.asarray(delta, dtype=float)
    n = np.sum(np.isfinite(delta), axis=0)
    mean = np.nanmean(delta, axis=0)
    sem = stats.sem(delta, axis=0, nan_policy="omit")
    critical = stats.t.ppf(0.975, np.maximum(n - 1, 1))
    return mean - critical * sem, mean + critical * sem


def run_arm_prepost(arm, context, output_dir, include_hdrs=True):
    output_dir.mkdir(parents=True, exist_ok=True)
    arm_clinical = clinical[
        (clinical["analysis_cohort"] == "TRD2")
        & (clinical["analysis_arm"] == arm)
    ].copy()
    required = ROI_COLUMNS + ["age", "sex", "education", "hdrs_improvement"]
    pre = arm_clinical.merge(pet_pre, on="subject_key", how="inner").dropna(subset=required)
    post = arm_clinical.merge(pet_post, on="subject_key", how="inner").dropna(subset=required)
    paired = sorted(set(pre["subject_key"]).intersection(post["subject_key"]))
    pre = pre[pre["subject_key"].isin(paired)].sort_values("subject_key").reset_index(drop=True)
    post = post[post["subject_key"].isin(paired)].sort_values("subject_key").reset_index(drop=True)
    assert len(pre) == EXPECTED_PAIRED_ARMS[arm]
    assert pre["subject_key"].tolist() == post["subject_key"].tolist()

    edge_pre = individual_edge_deviation(
        pre,
        context["reference_values"],
        context["reference_covariates"],
        context["pcc_reference"],
        context["denominator"],
        context["reference_covariate_names"],
    )
    edge_post = individual_edge_deviation(
        post,
        context["reference_values"],
        context["reference_covariates"],
        context["pcc_reference"],
        context["denominator"],
        context["reference_covariate_names"],
    )
    edge_delta = edge_post - edge_pre
    t_edge, p_edge = stats.ttest_rel(edge_post, edge_pre, axis=0, nan_policy="omit")
    edge_ci_low, edge_ci_high = paired_mean_ci95(edge_delta)

    edge_all = pd.DataFrame({
        "ROI_1": [pair[0] for pair in CROSS_EDGE_NAMES],
        "ROI_2": [pair[1] for pair in CROSS_EDGE_NAMES],
        "pre_deviation": np.nanmean(edge_pre, axis=0),
        "pre_SD_deviation": np.nanstd(edge_pre, axis=0, ddof=1),
        "post_deviation": np.nanmean(edge_post, axis=0),
        "post_SD_deviation": np.nanstd(edge_post, axis=0, ddof=1),
        "post_minus_pre": np.nanmean(edge_delta, axis=0),
        "post_minus_pre_CI95_low": edge_ci_low,
        "post_minus_pre_CI95_high": edge_ci_high,
        "paired_t": t_edge,
        "effect_size_Cohen_dz": t_edge / np.sqrt(len(pre)),
        "paired_p_uncorrected": p_edge,
    }).sort_values("paired_p_uncorrected").reset_index(drop=True)
    edge_selected = edge_all[
        edge_all["paired_p_uncorrected"] < PREPOST_P_THRESHOLD
    ].copy()

    hdrs = pre["hdrs_improvement"].to_numpy(float)
    age = pre["age"].to_numpy(float)
    sex = pre["sex"].to_numpy(float)
    edge_lookup = {pair: index for index, pair in enumerate(CROSS_EDGE_NAMES)}
    edge_hdrs_all = add_hdrs_correlations(
        edge_selected if include_hdrs else edge_selected.iloc[0:0],
        edge_delta,
        lambda row: edge_lookup[(row["ROI_1"], row["ROI_2"])],
        hdrs,
        age,
        sex,
        "HDRS_q_fdr_BH_within_prepost_significant_edges",
    )
    edge_hdrs_fdr, edge_hdrs_exploratory = save_prepost_and_hdrs(
        edge_all,
        edge_selected,
        edge_hdrs_all,
        output_dir,
        "edge",
        "HDRS_q_fdr_BH_within_prepost_significant_edges",
        include_hdrs=include_hdrs,
    )

    pre_design = sm.add_constant(pre[["age", "sex"]], has_constant="add").reindex(
        columns=context["regional_hc_design_columns"]
    )
    post_design = sm.add_constant(post[["age", "sex"]], has_constant="add").reindex(
        columns=context["regional_hc_design_columns"]
    )
    regional_pre = (
        pre[ROI_COLUMNS].to_numpy(float) - pre_design.to_numpy(float) @ context["regional_beta"]
    ) / context["reference_sd"].to_numpy(float)
    regional_post = (
        post[ROI_COLUMNS].to_numpy(float) - post_design.to_numpy(float) @ context["regional_beta"]
    ) / context["reference_sd"].to_numpy(float)
    regional_delta = regional_post - regional_pre
    t_regional, p_regional = stats.ttest_rel(
        regional_post, regional_pre, axis=0, nan_policy="omit"
    )
    regional_ci_low, regional_ci_high = paired_mean_ci95(regional_delta)
    regional_all = pd.DataFrame({
        "ROI": ROI_COLUMNS,
        "pre_deviation": np.nanmean(regional_pre, axis=0),
        "pre_SD_deviation": np.nanstd(regional_pre, axis=0, ddof=1),
        "post_deviation": np.nanmean(regional_post, axis=0),
        "post_SD_deviation": np.nanstd(regional_post, axis=0, ddof=1),
        "post_minus_pre": np.nanmean(regional_delta, axis=0),
        "post_minus_pre_CI95_low": regional_ci_low,
        "post_minus_pre_CI95_high": regional_ci_high,
        "paired_t": t_regional,
        "effect_size_Cohen_dz": t_regional / np.sqrt(len(pre)),
        "paired_p_uncorrected": p_regional,
    }).sort_values("paired_p_uncorrected").reset_index(drop=True)
    regional_selected = regional_all[
        regional_all["paired_p_uncorrected"] < PREPOST_P_THRESHOLD
    ].copy()
    regional_lookup = {roi: index for index, roi in enumerate(ROI_COLUMNS)}
    regional_hdrs_all = add_hdrs_correlations(
        regional_selected if include_hdrs else regional_selected.iloc[0:0],
        regional_delta,
        lambda row: regional_lookup[row["ROI"]],
        hdrs,
        age,
        sex,
        "HDRS_q_fdr_BH_within_prepost_significant_regions",
    )
    regional_hdrs_fdr, regional_hdrs_exploratory = save_prepost_and_hdrs(
        regional_all,
        regional_selected,
        regional_hdrs_all,
        output_dir,
        "regional",
        "HDRS_q_fdr_BH_within_prepost_significant_regions",
        include_hdrs=include_hdrs,
    )

    return {
        "arm": arm,
        "n": len(pre),
        "edge_all": edge_all,
        "edge_selected": edge_selected,
        "edge_hdrs_all": edge_hdrs_all,
        "edge_hdrs_fdr": edge_hdrs_fdr,
        "edge_hdrs_exploratory": edge_hdrs_exploratory,
        "regional_all": regional_all,
        "regional_selected": regional_selected,
        "regional_hdrs_all": regional_hdrs_all,
        "regional_hdrs_fdr": regional_hdrs_fdr,
        "regional_hdrs_exploratory": regional_hdrs_exploratory,
        "subject_keys": pre["subject_key"].to_numpy(),
        "hdrs_improvement": hdrs,
        "age": age,
        "sex": sex,
        "education": pre["education"].to_numpy(float),
        "edge_pre": edge_pre,
        "edge_post": edge_post,
        "edge_delta": edge_delta,
        "regional_pre": regional_pre,
        "regional_post": regional_post,
        "regional_delta": regional_delta,
    }


itbs_result = run_arm_prepost("iTBS", trd2_context, TRD2_OUT / "03_iTBS_prepost")
rtms_result = run_arm_prepost("rTMS", trd2_context, TRD2_OUT / "04_rTMS_prepost")
sham_result = run_arm_prepost("sham", trd2_context, TRD2_OUT / "05_sham_prepost", include_hdrs=False)

LONGITUDINAL_EDGE_FILE = TRD2_OUT / "longitudinal_high_quality_PET_subject_matrices_cross168.npz"
_paired_results = [itbs_result, rtms_result, sham_result]
np.savez_compressed(
    LONGITUDINAL_EDGE_FILE,
    subject_keys=np.concatenate([result["subject_keys"] for result in _paired_results]),
    arms=np.concatenate([np.repeat(result["arm"], result["n"]) for result in _paired_results]),
    edge_pre=np.vstack([result["edge_pre"] for result in _paired_results]),
    edge_post=np.vstack([result["edge_post"] for result in _paired_results]),
    edge_roi1=np.asarray([edge[0] for edge in CROSS_EDGE_NAMES], dtype=object),
    edge_roi2=np.asarray([edge[1] for edge in CROSS_EDGE_NAMES], dtype=object),
)
assert sum(EXPECTED_PAIRED_ARMS.values()) == sum(result["n"] for result in _paired_results)
print("Longitudinal edge archive:", LONGITUDINAL_EDGE_FILE)


prepost_counts = pd.DataFrame([
    {
        "arm": result["arm"],
        "n": result["n"],
        "edge_prepost_uncorrected_p_lt_0p05": len(result["edge_selected"]),
        "edge_HDRS_FDR_q_lt_0p05_primary": len(result["edge_hdrs_fdr"]),
        "edge_HDRS_uncorrected_p_lt_0p05_exploratory": len(result["edge_hdrs_exploratory"]),
        "regional_prepost_uncorrected_p_lt_0p05": len(result["regional_selected"]),
        "regional_HDRS_FDR_q_lt_0p05_primary": len(result["regional_hdrs_fdr"]),
        "regional_HDRS_uncorrected_p_lt_0p05_exploratory": len(result["regional_hdrs_exploratory"]),
    }
    for result in [itbs_result, rtms_result, sham_result]
])
prepost_counts.to_excel(TRD2_OUT / "prepost_and_HDRS_summary_counts.xlsx", index=False)
display(prepost_counts)


Longitudinal edge archive: ./outputs/main_analyses/02_TRD2_deviation_prepost_agesex/longitudinal_high_quality_PET_subject_matrices_cross168.npz


,arm,n,edge_prepost_uncorrected_p_lt_0p05,edge_HDRS_FDR_q_lt_0p05_primary,edge_HDRS_uncorrected_p_lt_0p05_exploratory,regional_prepost_uncorrected_p_lt_0p05,regional_HDRS_FDR_q_lt_0p05_primary,regional_HDRS_uncorrected_p_lt_0p05_exploratory
0,iTBS,37,12,0,0,0,0,0
1,rTMS,34,10,0,1,0,0,0
2,sham,33,7,0,0,8,0,0


## Results preview

Key numerical tables and figures generated by this step are displayed below. The full-resolution files remain under `outputs/`.


In [4]:
from io import BytesIO
import json as _preview_json
import pandas as _preview_pd
from PIL import Image as PILImage
from IPython.display import Image as NotebookImage, Markdown, display

_inline_table_specs = [{'title': 'Longitudinal sample and result counts', 'path': 'outputs/main_analyses/02_TRD2_deviation_prepost_agesex/prepost_and_HDRS_summary_counts.xlsx'}, {'title': 'iTBS connectivity changes used for figures (paired p < .01)', 'path': 'outputs/main_analyses/02_TRD2_deviation_prepost_agesex/03_iTBS_prepost/edge_pairttest_all.xlsx', 'query': 'paired_p_uncorrected < 0.01'}, {'title': 'rTMS connectivity changes used for figures (paired p < .01)', 'path': 'outputs/main_analyses/02_TRD2_deviation_prepost_agesex/04_rTMS_prepost/edge_pairttest_all.xlsx', 'query': 'paired_p_uncorrected < 0.01'}, {'title': 'iTBS HDRS-17 associations', 'path': 'outputs/main_analyses/02_TRD2_deviation_prepost_agesex/03_iTBS_prepost/edge_HDRS_partial_spearman_all_prepost_p_lt_0p05.xlsx', 'max_rows': 20}, {'title': 'rTMS HDRS-17 associations', 'path': 'outputs/main_analyses/02_TRD2_deviation_prepost_agesex/04_rTMS_prepost/edge_HDRS_partial_spearman_all_prepost_p_lt_0p05.xlsx', 'max_rows': 20}]
_inline_figure_paths = []

display(Markdown("### Key tables"))
for _table_spec in _inline_table_specs:
    _relative_table = _table_spec["path"]
    _table_path = _repo / _relative_table
    if not _table_path.is_file():
        raise FileNotFoundError(_table_path)
    if _table_path.suffix.lower() == ".csv":
        _preview_table = _preview_pd.read_csv(_table_path)
    elif _table_path.suffix.lower() in {".xlsx", ".xls"}:
        _preview_table = _preview_pd.read_excel(
            _table_path, sheet_name=_table_spec.get("sheet", 0)
        )
    elif _table_path.suffix.lower() == ".json":
        _payload = _preview_json.loads(_table_path.read_text(encoding="utf-8"))
        _checks = _payload.get("checks", _payload)
        _preview_table = _preview_pd.DataFrame(
            {"item": list(_checks), "value": list(_checks.values())}
        ) if isinstance(_checks, dict) else _preview_pd.json_normalize(_payload)
    else:
        raise ValueError(f"Unsupported table format: {_table_path.suffix}")
    if _table_spec.get("query"):
        _preview_table = _preview_table.query(_table_spec["query"]).reset_index(drop=True)
    _max_rows = int(_table_spec.get("max_rows", 40))
    display(Markdown(
        f"**{_table_spec['title']}**  \n"
        f"`{_relative_table}` — {len(_preview_table)} rows × {len(_preview_table.columns)} columns"
    ))
    with _preview_pd.option_context("display.max_columns", 40, "display.max_colwidth", 60):
        display(_preview_table.head(_max_rows))

if _inline_figure_paths:
    display(Markdown("### Figures"))
for _relative_figure in _inline_figure_paths:
    _figure_path = _repo / _relative_figure
    if not _figure_path.is_file():
        raise FileNotFoundError(_figure_path)
    with PILImage.open(_figure_path) as _source_image:
        _preview_image = _source_image.convert("RGB")
        _preview_image.thumbnail((1600, 1200), PILImage.Resampling.LANCZOS)
        _preview_buffer = BytesIO()
        _preview_image.save(_preview_buffer, format="PNG", optimize=True)
    display(Markdown(f"**`{_relative_figure}`**"))
    display(NotebookImage(data=_preview_buffer.getvalue(), format="png"))


### Key tables

**Longitudinal sample and result counts**  
`outputs/main_analyses/02_TRD2_deviation_prepost_agesex/prepost_and_HDRS_summary_counts.xlsx` — 3 rows × 8 columns

,arm,n,edge_prepost_uncorrected_p_lt_0p05,edge_HDRS_FDR_q_lt_0p05_primary,edge_HDRS_uncorrected_p_lt_0p05_exploratory,regional_prepost_uncorrected_p_lt_0p05,regional_HDRS_FDR_q_lt_0p05_primary,regional_HDRS_uncorrected_p_lt_0p05_exploratory
0,iTBS,37,12,0,0,0,0,0
1,rTMS,34,10,0,1,0,0,0
2,sham,33,7,0,0,8,0,0


**iTBS connectivity changes used for figures (paired p < .01)**  
`outputs/main_analyses/02_TRD2_deviation_prepost_agesex/03_iTBS_prepost/edge_pairttest_all.xlsx` — 3 rows × 12 columns

,ROI_1,ROI_2,pre_deviation,pre_SD_deviation,post_deviation,post_SD_deviation,post_minus_pre,post_minus_pre_CI95_low,post_minus_pre_CI95_high,paired_t,effect_size_Cohen_dz,paired_p_uncorrected
0,AAL3_031_OFClat_L,AAL3_151_ACC_sub_L,0.322443,0.554042,-0.025557,0.658100,-0.348000,-0.572619,-0.123381,-3.142105,-0.516559,0.003349
1,AAL3_031_OFClat_L,AAL3_041_Hippocampus_L,-0.129263,0.638306,-0.393839,0.799275,-0.264576,-0.449840,-0.079312,-2.896328,-0.476153,0.006384
2,AAL3_019_Frontal_Sup_Medial_L,AAL3_079_Pallidum_L,-0.636765,2.321924,-0.061021,2.235979,0.575744,0.159609,0.991879,2.805974,0.461299,0.008043


...

,ROI_1,ROI_2,pre_deviation,pre_SD_deviation,post_deviation,post_SD_deviation,post_minus_pre,post_minus_pre_CI95_low,post_minus_pre_CI95_high,paired_t,effect_size_Cohen_dz,paired_p_uncorrected
0,AAL3_022_Frontal_Med_Orb_R,AAL3_078_Putamen_R,-0.474351,1.559057,0.218445,1.239077,0.692795,0.301172,1.084418,3.599133,0.617246,0.001033
1,AAL3_026_OFCmed_R,AAL3_078_Putamen_R,-0.273847,1.553380,0.514968,1.791705,0.788815,0.318728,1.258902,3.413956,0.585489,0.001712
2,AAL3_026_OFCmed_R,AAL3_080_Pallidum_R,-0.087030,1.158138,0.377345,1.419333,0.464375,0.132766,0.795984,2.849074,0.488612,0.007495


**iTBS HDRS-17 associations**  
`outputs/main_analyses/02_TRD2_deviation_prepost_agesex/03_iTBS_prepost/edge_HDRS_partial_spearman_all_prepost_p_lt_0p05.xlsx` — 12 rows × 18 columns

,ROI_1,ROI_2,pre_deviation,pre_SD_deviation,post_deviation,post_SD_deviation,post_minus_pre,post_minus_pre_CI95_low,post_minus_pre_CI95_high,paired_t,effect_size_Cohen_dz,paired_p_uncorrected,HDRS_partial_Spearman_r,HDRS_p_uncorrected,n_used,HDRS_q_fdr_BH_within_prepost_significant_edges,HDRS_primary_FDR_q_lt_0p05,HDRS_exploratory_uncorrected_p_lt_0p05
0,AAL3_022_Frontal_Med_Orb_R,AAL3_151_ACC_sub_L,-1.163232,2.160874,-2.042845,3.157369,-0.879613,-1.693157,-0.066070,-2.192800,-0.360494,0.034867,0.221740,0.200475,37,0.662729,False,False
1,AAL3_031_OFClat_L,AAL3_041_Hippocampus_L,-0.129263,0.638306,-0.393839,0.799275,-0.264576,-0.449840,-0.079312,-2.896328,-0.476153,0.006384,0.210360,0.225153,37,0.662729,False,False
2,AAL3_004_Frontal_Sup_2_R,AAL3_041_Hippocampus_L,-0.245270,0.978001,-0.495231,1.076867,-0.249961,-0.488444,-0.011478,-2.125705,-0.349464,0.040457,-0.191589,0.270225,37,0.662729,False,False
3,AAL3_019_Frontal_Sup_Medial_L,AAL3_080_Pallidum_R,-0.462256,1.974637,-0.005566,1.998026,0.456690,0.008258,0.905121,2.065443,0.339557,0.046138,0.186233,0.284096,37,0.662729,False,False
4,AAL3_021_Frontal_Med_Orb_L,AAL3_079_Pallidum_L,-0.543471,2.853608,-0.091971,2.588618,0.451500,0.006287,0.896713,2.056733,0.338125,0.047014,-0.166654,0.338644,37,0.662729,False,False
5,AAL3_031_OFClat_L,AAL3_151_ACC_sub_L,0.322443,0.554042,-0.025557,0.658100,-0.348000,-0.572619,-0.123381,-3.142105,-0.516559,0.003349,-0.151338,0.385489,37,0.662729,False,False
6,AAL3_020_Frontal_Sup_Medial_R,AAL3_151_ACC_sub_L,-0.674562,1.411727,-1.149253,1.918509,-0.474690,-0.944425,-0.004956,-2.049492,-0.336934,0.047754,0.150991,0.386592,37,0.662729,False,False
7,AAL3_020_Frontal_Sup_Medial_R,AAL3_079_Pallidum_L,-0.044957,2.364174,0.487212,2.767841,0.532169,0.024216,1.040121,2.124782,0.349312,0.040539,0.131353,0.451967,37,0.677951,False,False
8,AAL3_019_Frontal_Sup_Medial_L,AAL3_079_Pallidum_L,-0.636765,2.321924,-0.061021,2.235979,0.575744,0.159609,0.991879,2.805974,0.461299,0.008043,0.097622,0.576914,37,0.769219,False,False
9,AAL3_021_Frontal_Med_Orb_L,AAL3_151_ACC_sub_L,-0.521271,1.402799,-0.986941,1.946322,-0.465670,-0.868422,-0.062919,-2.344926,-0.385503,0.024663,-0.062174,0.722731,37,0.867277,False,False


**rTMS HDRS-17 associations**  
`outputs/main_analyses/02_TRD2_deviation_prepost_agesex/04_rTMS_prepost/edge_HDRS_partial_spearman_all_prepost_p_lt_0p05.xlsx` — 10 rows × 18 columns

,ROI_1,ROI_2,pre_deviation,pre_SD_deviation,post_deviation,post_SD_deviation,post_minus_pre,post_minus_pre_CI95_low,post_minus_pre_CI95_high,paired_t,effect_size_Cohen_dz,paired_p_uncorrected,HDRS_partial_Spearman_r,HDRS_p_uncorrected,n_used,HDRS_q_fdr_BH_within_prepost_significant_edges,HDRS_primary_FDR_q_lt_0p05,HDRS_exploratory_uncorrected_p_lt_0p05
0,AAL3_026_OFCmed_R,AAL3_080_Pallidum_R,-0.087030,1.158138,0.377345,1.419333,0.464375,0.132766,0.795984,2.849074,0.488612,0.007495,0.389852,0.027405,34,0.274054,False,True
1,AAL3_022_Frontal_Med_Orb_R,AAL3_080_Pallidum_R,0.473938,1.502926,1.053895,1.735894,0.579957,0.092463,1.067451,2.420400,0.415095,0.021170,0.205121,0.260080,34,0.786649,False,False
2,AAL3_022_Frontal_Med_Orb_R,AAL3_076_Caudate_R,-1.464076,2.427355,-0.708736,1.418019,0.755340,0.081443,1.429237,2.280395,0.391084,0.029175,0.204154,0.262383,34,0.786649,False,False
3,AAL3_021_Frontal_Med_Orb_L,AAL3_076_Caudate_R,-2.151426,3.422661,-1.091316,2.517018,1.060110,0.217641,1.902580,2.560106,0.439055,0.015228,0.146326,0.424217,34,0.786649,False,False
4,AAL3_026_OFCmed_R,AAL3_042_Hippocampus_R,-0.210529,0.642216,0.110255,0.930839,0.320783,0.080254,0.561313,2.713337,0.465334,0.010503,0.142704,0.435899,34,0.786649,False,False
5,AAL3_032_OFClat_R,AAL3_042_Hippocampus_R,-0.412694,0.658859,-0.179942,0.533307,0.232752,0.028317,0.437186,2.316321,0.397246,0.026895,0.131835,0.471989,34,0.786649,False,False
6,AAL3_022_Frontal_Med_Orb_R,AAL3_078_Putamen_R,-0.474351,1.559057,0.218445,1.239077,0.692795,0.301172,1.084418,3.599133,0.617246,0.001033,0.103936,0.571327,34,0.816182,False,False
7,AAL3_026_OFCmed_R,AAL3_078_Putamen_R,-0.273847,1.553380,0.514968,1.791705,0.788815,0.318728,1.258902,3.413956,0.585489,0.001712,0.066153,0.719055,34,0.848700,False,False
8,AAL3_026_OFCmed_R,AAL3_077_Putamen_L,-0.413277,1.332909,0.064399,1.631776,0.477676,0.042754,0.912599,2.234515,0.383216,0.032338,0.055272,0.763830,34,0.848700,False,False
9,AAL3_022_Frontal_Med_Orb_R,AAL3_077_Putamen_L,-0.424234,1.458776,0.082533,1.145038,0.506766,0.068711,0.944822,2.353637,0.403645,0.024698,0.022770,0.901553,34,0.901553,False,False
